
# Práctica: Redes Neuronales desde Cero

## De una neurona artificial al aprendizaje de un perceptrón

Esta práctica tiene como finalidad que el estudiante comprenda, de forma progresiva y visual, cómo funciona una red neuronal artificial desde sus elementos más básicos.

Se parte de una sola neurona artificial, se analiza el funcionamiento de un perceptrón, se estudian ejemplos linealmente separables como la clasificación de una manzana roja y redonda, se revisa por qué el problema XOR no puede resolverse con un solo perceptrón y finalmente se incorpora un bloque de entrenamiento para observar cómo una neurona aprende ajustando sus pesos y su bias.

Cada bloque contiene explicación conceptual, código en Python y simuladores interactivos.



## Instalación de bibliotecas

Antes de ejecutar este cuaderno, se recomienda instalar las siguientes bibliotecas en el entorno de Python:

```bash
pip install numpy matplotlib ipywidgets
```

En Jupyter Notebook o JupyterLab, los controles interactivos requieren `ipywidgets`.


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, FloatSlider, fixed
import itertools

plt.rcParams["figure.figsize"] = (6, 5)



# Bloque 1. Neurona artificial

Una neurona artificial recibe entradas numéricas, las multiplica por pesos, suma un valor llamado bias y aplica una función de activación.

Matemáticamente:

$$
z = w_1 x_1 + w_2 x_2 + b
$$

$$
y = f(z)
$$

De forma general, para una neurona con varias entradas:

$$
z = \sum_{i=1}^{n} w_i x_i + b
$$

Donde:

- \(x_1, x_2\): entradas.
- \(w_1, w_2\): pesos.
- \(b\): bias o sesgo.
- \(z\): suma ponderada.
- \(f(z)\): función de activación.
- \(y\): salida de la neurona.

En este primer ejemplo se usará una función escalón:

$$
f(z) =
\begin{cases}
1 & \text{si } z \geq 0 \\
0 & \text{si } z < 0
\end{cases}
$$


In [ ]:

def activacion_escalon(z):
    return 1 if z >= 0 else 0

def neurona(x1, x2, w1, w2, b):
    z = w1*x1 + w2*x2 + b
    y = activacion_escalon(z)
    return z, y

def simulador_neurona(x1=1.0, x2=1.0, w1=1.0, w2=1.0, b=0.0):
    z, y = neurona(x1, x2, w1, w2, b)

    print("Entradas:")
    print(f"x1 = {x1}")
    print(f"x2 = {x2}")
    print("\nParámetros:")
    print(f"w1 = {w1}")
    print(f"w2 = {w2}")
    print(f"b  = {b}")
    print("\nCálculo:")
    print(f"z = ({w1})({x1}) + ({w2})({x2}) + ({b}) = {z:.3f}")
    print(f"Salida y = {y}")

interact(
    simulador_neurona,
    x1=FloatSlider(value=1, min=-5, max=5, step=0.1),
    x2=FloatSlider(value=1, min=-5, max=5, step=0.1),
    w1=FloatSlider(value=1, min=-5, max=5, step=0.1),
    w2=FloatSlider(value=1, min=-5, max=5, step=0.1),
    b=FloatSlider(value=0, min=-5, max=5, step=0.1)
);



# Bloque 2. Frontera de decisión de un perceptrón

Cuando una neurona tiene dos entradas, se puede visualizar su decisión como una recta en el plano.

La frontera de decisión ocurre cuando:

$$
w_1 x_1 + w_2 x_2 + b = 0
$$

Despejando \(x_2\):

$$
x_2 = -\frac{w_1 x_1 + b}{w_2}
$$

La recta separa el plano en dos regiones:

- Región donde la salida es 0.
- Región donde la salida es 1.


In [ ]:

def graficar_frontera(w1=1.0, w2=1.0, b=-1.0):
    x = np.linspace(-5, 5, 200)

    plt.figure()

    if abs(w2) > 1e-9:
        y = -(w1*x + b)/w2
        plt.plot(x, y, label="Frontera de decisión")
    else:
        if abs(w1) > 1e-9:
            x_vertical = -b/w1
            plt.axvline(x_vertical, label="Frontera de decisión")
        else:
            plt.text(-4, 0, "No hay frontera definida")

    xx, yy = np.meshgrid(np.linspace(-5, 5, 100), np.linspace(-5, 5, 100))
    zz = w1*xx + w2*yy + b
    plt.contourf(xx, yy, zz >= 0, alpha=0.25)

    plt.axhline(0)
    plt.axvline(0)
    plt.xlim(-5, 5)
    plt.ylim(-5, 5)
    plt.grid(True)
    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.title("Frontera de decisión del perceptrón")
    plt.legend()
    plt.show()

interact(
    graficar_frontera,
    w1=FloatSlider(value=1, min=-5, max=5, step=0.1),
    w2=FloatSlider(value=1, min=-5, max=5, step=0.1),
    b=FloatSlider(value=-1, min=-5, max=5, step=0.1)
);



# Bloque 3. Ejemplo: clasificación de una manzana

Se desea clasificar si un objeto puede considerarse una manzana usando dos características binarias:

| Característica | Valor 0 | Valor 1 |
|---|---|---|
| Color | No rojo | Rojo |
| Forma | No redonda | Redonda |

Criterio:

- Si el objeto es rojo y redondo, se clasifica como manzana.
- En caso contrario, no se clasifica como manzana.

Este problema sí puede ser resuelto por un perceptrón porque es linealmente separable.


In [ ]:

datos_manzana = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
])

etiquetas_manzana = np.array([0, 0, 0, 1])

def clasificar_manzana(color_rojo, forma_redonda, w1=1, w2=1, b=-1.5):
    z, y = neurona(color_rojo, forma_redonda, w1, w2, b)
    print(f"Color rojo: {color_rojo}")
    print(f"Forma redonda: {forma_redonda}")
    print(f"z = {z:.2f}")
    print(f"Clasificación: {y}")

    if y == 1:
        print("Resultado: el objeto se clasifica como manzana.")
    else:
        print("Resultado: el objeto NO se clasifica como manzana.")

interact(
    clasificar_manzana,
    color_rojo=IntSlider(value=1, min=0, max=1, step=1),
    forma_redonda=IntSlider(value=1, min=0, max=1, step=1),
    w1=FloatSlider(value=1, min=-5, max=5, step=0.1),
    w2=FloatSlider(value=1, min=-5, max=5, step=0.1),
    b=FloatSlider(value=-1.5, min=-5, max=5, step=0.1)
);


In [ ]:

def graficar_manzana(w1=1.0, w2=1.0, b=-1.5):
    plt.figure()

    for punto, etiqueta in zip(datos_manzana, etiquetas_manzana):
        marcador = "o" if etiqueta == 1 else "x"
        plt.scatter(punto[0], punto[1], s=150, marker=marcador, label=f"Clase {etiqueta}" if f"Clase {etiqueta}" not in plt.gca().get_legend_handles_labels()[1] else "")
        plt.text(punto[0]+0.03, punto[1]+0.03, f"{punto} -> {etiqueta}")

    x = np.linspace(-0.5, 1.5, 100)
    if abs(w2) > 1e-9:
        y = -(w1*x + b)/w2
        plt.plot(x, y, label="Frontera")

    plt.xlim(-0.5, 1.5)
    plt.ylim(-0.5, 1.5)
    plt.xticks([0, 1])
    plt.yticks([0, 1])
    plt.xlabel("Color rojo")
    plt.ylabel("Forma redonda")
    plt.grid(True)
    plt.title("Clasificación lineal: manzana")
    plt.legend()
    plt.show()

interact(
    graficar_manzana,
    w1=FloatSlider(value=1, min=-5, max=5, step=0.1),
    w2=FloatSlider(value=1, min=-5, max=5, step=0.1),
    b=FloatSlider(value=-1.5, min=-5, max=5, step=0.1)
);



# Bloque 4. Problema XOR

El problema XOR tiene la siguiente tabla de verdad:

| \(x_1\) | \(x_2\) | Salida |
|---|---|---|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

Este problema no puede resolverse con un solo perceptrón porque sus clases no son linealmente separables.

No existe una sola recta que separe correctamente los puntos con salida 1 de los puntos con salida 0.


In [ ]:

datos_xor = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
])

etiquetas_xor = np.array([0, 1, 1, 0])

def probar_xor_con_un_perceptron(w1=1.0, w2=1.0, b=-0.5):
    errores = 0

    print("Prueba XOR con un solo perceptrón")
    print("--------------------------------")

    for (x1, x2), esperado in zip(datos_xor, etiquetas_xor):
        z, salida = neurona(x1, x2, w1, w2, b)
        error = esperado - salida
        errores += abs(error)
        print(f"x1={x1}, x2={x2} | esperado={esperado}, obtenido={salida}, error={error}")

    print("--------------------------------")
    print(f"Errores totales: {errores}")

interact(
    probar_xor_con_un_perceptron,
    w1=FloatSlider(value=1, min=-5, max=5, step=0.1),
    w2=FloatSlider(value=1, min=-5, max=5, step=0.1),
    b=FloatSlider(value=-0.5, min=-5, max=5, step=0.1)
);


In [ ]:

def graficar_xor(w1=1.0, w2=1.0, b=-0.5):
    plt.figure()

    for punto, etiqueta in zip(datos_xor, etiquetas_xor):
        marcador = "o" if etiqueta == 1 else "x"
        plt.scatter(punto[0], punto[1], s=150, marker=marcador, label=f"Clase {etiqueta}" if f"Clase {etiqueta}" not in plt.gca().get_legend_handles_labels()[1] else "")
        plt.text(punto[0]+0.03, punto[1]+0.03, f"{punto} -> {etiqueta}")

    x = np.linspace(-0.5, 1.5, 100)
    if abs(w2) > 1e-9:
        y = -(w1*x + b)/w2
        plt.plot(x, y, label="Frontera propuesta")

    plt.xlim(-0.5, 1.5)
    plt.ylim(-0.5, 1.5)
    plt.xticks([0, 1])
    plt.yticks([0, 1])
    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.grid(True)
    plt.title("XOR no es linealmente separable")
    plt.legend()
    plt.show()

interact(
    graficar_xor,
    w1=FloatSlider(value=1, min=-5, max=5, step=0.1),
    w2=FloatSlider(value=1, min=-5, max=5, step=0.1),
    b=FloatSlider(value=-0.5, min=-5, max=5, step=0.1)
);



# Bloque 5. Red multicapa para resolver XOR

Para resolver XOR se necesita combinar varias neuronas.

Una forma intuitiva es construir dos neuronas en una capa oculta y una neurona de salida.

La red puede formar regiones de decisión más complejas que una sola recta.

En este ejemplo se construye una red equivalente a:

$$
\text{XOR}(x_1, x_2) = \text{OR}(x_1, x_2) \land \neg(\text{AND}(x_1, x_2))
$$


In [ ]:

def perceptron_binario(x1, x2, w1, w2, b):
    return activacion_escalon(w1*x1 + w2*x2 + b)

def red_xor_manual(x1, x2):
    # Neurona oculta 1: OR
    h1 = perceptron_binario(x1, x2, 1, 1, -0.5)

    # Neurona oculta 2: AND
    h2 = perceptron_binario(x1, x2, 1, 1, -1.5)

    # Neurona de salida: h1 AND NOT h2
    y = perceptron_binario(h1, h2, 1, -2, -0.5)

    return h1, h2, y

for x1, x2 in datos_xor:
    h1, h2, y = red_xor_manual(x1, x2)
    print(f"x1={x1}, x2={x2} | h1={h1}, h2={h2} | salida XOR={y}")


In [ ]:

def simulador_red_xor(x1=0, x2=0):
    h1, h2, y = red_xor_manual(x1, x2)

    print("Entradas")
    print(f"x1 = {x1}")
    print(f"x2 = {x2}")
    print("\nCapa oculta")
    print(f"h1 = OR(x1, x2)  = {h1}")
    print(f"h2 = AND(x1, x2) = {h2}")
    print("\nSalida")
    print(f"XOR = {y}")

interact(
    simulador_red_xor,
    x1=IntSlider(value=0, min=0, max=1, step=1),
    x2=IntSlider(value=0, min=0, max=1, step=1)
);



# Bloque 6. Entrenamiento de un perceptrón

Hasta este punto se han colocado los pesos manualmente. Ahora se mostrará cómo un perceptrón puede aprender.

El entrenamiento consiste en ajustar los pesos y el bias cuando la salida obtenida es diferente de la salida esperada.

La regla de aprendizaje del perceptrón es:

$$
w_i \leftarrow w_i + \eta \cdot \text{error} \cdot x_i
$$

$$
b \leftarrow b + \eta \cdot \text{error}
$$

Donde:

- \(\eta\): tasa de aprendizaje.
- \(\text{error} = y_{\text{esperado}} - y_{\text{obtenido}}\).


In [ ]:

def entrenar_perceptron(X, y, lr=0.1, epocas=10, w_inicial=None, b_inicial=0.0):
    if w_inicial is None:
        w = np.zeros(X.shape[1], dtype=float)
    else:
        w = np.array(w_inicial, dtype=float)

    b = float(b_inicial)
    historial = []

    for epoca in range(epocas):
        errores = 0

        for xi, objetivo in zip(X, y):
            z = np.dot(w, xi) + b
            salida = activacion_escalon(z)
            error = objetivo - salida

            w = w + lr * error * xi
            b = b + lr * error

            errores += abs(error)

        historial.append({
            "epoca": epoca + 1,
            "w": w.copy(),
            "b": b,
            "errores": errores
        })

    return w, b, historial

def visualizar_entrenamiento_manzana(lr=0.2, epocas=10, paso=1):
    w, b, historial = entrenar_perceptron(
        datos_manzana,
        etiquetas_manzana,
        lr=lr,
        epocas=epocas,
        w_inicial=[0.0, 0.0],
        b_inicial=0.0
    )

    paso = min(max(1, paso), len(historial))
    estado = historial[paso-1]
    w_actual = estado["w"]
    b_actual = estado["b"]

    print(f"Época mostrada: {estado['epoca']}")
    print(f"w1 = {w_actual[0]:.3f}")
    print(f"w2 = {w_actual[1]:.3f}")
    print(f"b  = {b_actual:.3f}")
    print(f"Errores en esta época: {estado['errores']}")

    plt.figure()

    for punto, etiqueta in zip(datos_manzana, etiquetas_manzana):
        marcador = "o" if etiqueta == 1 else "x"
        plt.scatter(punto[0], punto[1], s=150, marker=marcador)
        plt.text(punto[0]+0.03, punto[1]+0.03, f"{punto} -> {etiqueta}")

    x = np.linspace(-0.5, 1.5, 100)
    if abs(w_actual[1]) > 1e-9:
        y_linea = -(w_actual[0]*x + b_actual)/w_actual[1]
        plt.plot(x, y_linea, label="Frontera aprendida")
    elif abs(w_actual[0]) > 1e-9:
        plt.axvline(-b_actual/w_actual[0], label="Frontera aprendida")

    plt.xlim(-0.5, 1.5)
    plt.ylim(-0.5, 1.5)
    plt.xticks([0, 1])
    plt.yticks([0, 1])
    plt.xlabel("Color rojo")
    plt.ylabel("Forma redonda")
    plt.title("Entrenamiento del perceptrón para clasificar manzanas")
    plt.grid(True)
    plt.legend()
    plt.show()

    errores = [h["errores"] for h in historial]
    plt.figure()
    plt.plot(range(1, epocas+1), errores, marker="o")
    plt.xlabel("Época")
    plt.ylabel("Errores")
    plt.title("Errores durante el entrenamiento")
    plt.grid(True)
    plt.show()

interact(
    visualizar_entrenamiento_manzana,
    lr=FloatSlider(value=0.2, min=0.01, max=1.0, step=0.01),
    epocas=IntSlider(value=10, min=1, max=30, step=1),
    paso=IntSlider(value=1, min=1, max=30, step=1)
);



# Bloque 7. Intento de entrenamiento con XOR

Ahora se intenta entrenar un solo perceptrón para resolver XOR.

El resultado esperado es que el perceptrón no logre converger, porque XOR no es linealmente separable.

Esto permite comprobar experimentalmente la limitación del perceptrón simple.


In [ ]:

def visualizar_entrenamiento_xor(lr=0.2, epocas=20, paso=1):
    w, b, historial = entrenar_perceptron(
        datos_xor,
        etiquetas_xor,
        lr=lr,
        epocas=epocas,
        w_inicial=[0.0, 0.0],
        b_inicial=0.0
    )

    paso = min(max(1, paso), len(historial))
    estado = historial[paso-1]
    w_actual = estado["w"]
    b_actual = estado["b"]

    print(f"Época mostrada: {estado['epoca']}")
    print(f"w1 = {w_actual[0]:.3f}")
    print(f"w2 = {w_actual[1]:.3f}")
    print(f"b  = {b_actual:.3f}")
    print(f"Errores en esta época: {estado['errores']}")

    plt.figure()

    for punto, etiqueta in zip(datos_xor, etiquetas_xor):
        marcador = "o" if etiqueta == 1 else "x"
        plt.scatter(punto[0], punto[1], s=150, marker=marcador)
        plt.text(punto[0]+0.03, punto[1]+0.03, f"{punto} -> {etiqueta}")

    x = np.linspace(-0.5, 1.5, 100)
    if abs(w_actual[1]) > 1e-9:
        y_linea = -(w_actual[0]*x + b_actual)/w_actual[1]
        plt.plot(x, y_linea, label="Frontera intentada")
    elif abs(w_actual[0]) > 1e-9:
        plt.axvline(-b_actual/w_actual[0], label="Frontera intentada")

    plt.xlim(-0.5, 1.5)
    plt.ylim(-0.5, 1.5)
    plt.xticks([0, 1])
    plt.yticks([0, 1])
    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.title("Intento de entrenamiento de XOR con un perceptrón")
    plt.grid(True)
    plt.legend()
    plt.show()

    errores = [h["errores"] for h in historial]
    plt.figure()
    plt.plot(range(1, epocas+1), errores, marker="o")
    plt.xlabel("Época")
    plt.ylabel("Errores")
    plt.title("Errores durante el entrenamiento de XOR")
    plt.grid(True)
    plt.show()

interact(
    visualizar_entrenamiento_xor,
    lr=FloatSlider(value=0.2, min=0.01, max=1.0, step=0.01),
    epocas=IntSlider(value=20, min=1, max=50, step=1),
    paso=IntSlider(value=1, min=1, max=50, step=1)
);



# Actividad de comprobación

El estudiante deberá realizar las siguientes modificaciones:

1. Cambiar los pesos y el bias del perceptrón de manzana y observar cuándo clasifica correctamente.
2. Modificar la tasa de aprendizaje del entrenamiento y observar si aprende más rápido o más lento.
3. Ejecutar el entrenamiento con el conjunto XOR y explicar por qué el error no llega de forma estable a cero.
4. Proponer otro ejemplo con dos características binarias que sí pueda resolverse con un perceptrón.
5. Proponer otro ejemplo que no pueda resolverse con un solo perceptrón.

## Conclusión

Un perceptrón puede resolver problemas linealmente separables, como la clasificación básica de una manzana usando dos características simples. Sin embargo, no puede resolver problemas como XOR, porque requieren separar regiones que no pueden dividirse con una sola recta.

Para resolver problemas más complejos se requieren redes multicapa, funciones de activación diferenciables y algoritmos de entrenamiento más avanzados, como backpropagation.
